<a href="https://colab.research.google.com/github/MuzzamilRauf/dcgan-fashion-mnist/blob/main/dcgan_fashion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
import os
import sys
from pathlib import Path

# Identify project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    colab_project = Path("/content/dcgan-fashion-mnist")

    if (colab_project / "src").exists():
        PROJECT_ROOT = colab_project
    else:
        raise FileNotFoundError(
            "Project root not found. Open the notebook from the repository root."
        )

# Set working directory
os.chdir(PROJECT_ROOT)

# Add src directory to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Project setup completed!")

In [ ]:
import torch
import torchvision

# Display library versions
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# Check GPU availability
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
from generator import Generator
from discriminator import Discriminator

# Import training module
import train

print("Generator imported successfully!")
print("Discriminator imported successfully!")
print("Training module imported successfully!")

In [ ]:
import matplotlib.pyplot as plt

# Get the dataset loaded by train.py
train_dataset = train.train_dataset

# Fashion-MNIST class names
classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Display 10 sample images
plt.figure(figsize=(12, 5))

for i in range(10):
    image, label = train_dataset[i]

    plt.subplot(2, 5, i + 1)
    plt.imshow(image.squeeze(), cmap="gray")
    plt.title(classes[label])
    plt.axis("off")

plt.suptitle("Fashion-MNIST Real Images")
plt.tight_layout()
plt.show()

In [ ]:
# Get one batch of training data
real_images, labels = next(iter(train.train_loader))

print("Image batch shape:", real_images.shape)
print("Label batch shape:", labels.shape)

print("Minimum pixel value:", real_images.min().item())
print("Maximum pixel value:", real_images.max().item())

In [ ]:
# Select device
device = train.device

# Initialize Generator
generator = Generator(latent_dim=100).to(device)

# Generate random noise
noise = torch.randn(64, 100, device=device)

# Generate fake images
with torch.no_grad():
    fake_images = generator(noise)

print("Noise shape:", noise.shape)
print("Generated images shape:", fake_images.shape)
print("Minimum pixel value:", fake_images.min().item())
print("Maximum pixel value:", fake_images.max().item())

In [ ]:
# Initialize Discriminator
discriminator = Discriminator().to(device)

# Get predictions for fake images
with torch.no_grad():
    predictions = discriminator(fake_images)

print("Discriminator input shape:", fake_images.shape)
print("Discriminator output shape:", predictions.shape)

print("Sample logits:")
print(predictions[:5].flatten())

In [ ]:
from torchvision.utils import make_grid

# Create image grid
grid = make_grid(
    fake_images.cpu(),
    nrow=8,
    normalize=True,
    value_range=(-1, 1)
)

# Display generated images
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("Generated Images Before Training")
plt.axis("off")
plt.show()

In [ ]:
# Set initial test training duration
train.EPOCHS = 1

print("Training epochs:", train.EPOCHS)
print("Batch size:", train.BATCH_SIZE)
print("Learning rate:", train.LEARNING_RATE)
print("Training device:", train.device)

In [ ]:
# Start DCGAN training
train.train()

In [ ]:
from PIL import Image

# Locate generated image
image_path = PROJECT_ROOT / "outputs" / "epoch_1.png"

# Load generated image
generated_image = Image.open(image_path)

# Display result
plt.figure(figsize=(8, 8))
plt.imshow(generated_image, cmap="gray")
plt.title("DCGAN Generated Images After 1 Epoch")
plt.axis("off")
plt.show()

In [ ]:
# Configure full training
train.EPOCHS = 50

# Start full training
train.train()

In [ ]:
# Load images generated during the final epoch
final_image_path = PROJECT_ROOT / "outputs" / "epoch_50.png"

final_image = Image.open(final_image_path)

# Display final results
plt.figure(figsize=(8, 8))
plt.imshow(final_image, cmap="gray")
plt.title("DCGAN Fashion-MNIST - Final Generated Images")
plt.axis("off")
plt.show()

In [ ]:
# Initialize a fresh Generator
trained_generator = Generator(latent_dim=100).to(device)

# Load trained Generator weights
checkpoint_path = PROJECT_ROOT / "checkpoints" / "generator_final.pth"

trained_generator.load_state_dict(
    torch.load(checkpoint_path, map_location=device, weights_only=True)
)

# Generate new images
trained_generator.eval()

with torch.no_grad():
    new_noise = torch.randn(64, 100, device=device)
    new_images = trained_generator(new_noise)

print("New generated images:", new_images.shape)

In [ ]:
# Create image grid
new_grid = make_grid(
    new_images.cpu(),
    nrow=8,
    normalize=True,
    value_range=(-1, 1)
)

# Display newly generated samples
plt.figure(figsize=(8, 8))
plt.imshow(new_grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("New Images from Trained Generator")
plt.axis("off")
plt.show()